# From a bioframe result to a track

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/02_dataframe_analysis.ipynb)

[bioframe](https://bioframe.readthedocs.io) is the pandas-native toolkit for genomic intervals, and a bioframe frame is just a DataFrame with `chrom`/`start`/`end`. That's exactly what `add_features` takes — so any interval analysis you already do in bioframe is **one call from the genome**, no file written.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy bioframe

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Real intervals, one real operation

UCSC's hg38 **CpG islands** (read straight from UCSC with pandas), then their **shores** — the 2 kb flanks where most differential methylation sits. In bioframe that's `expand` minus the islands, one line. This assembly names chromosomes `17` (no `chr`), so match it.

In [ ]:
import bioframe as bf
import pandas as pd

cols = "bin chrom start end name length cpgNum gcNum perCpg perGc obsExp".split()
islands = pd.read_csv(
    "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cpgIslandExt.txt.gz",
    sep="\t", names=cols,
)
islands = islands[islands.chrom == "chr17"].assign(chrom="17")
shores = bf.merge(bf.subtract(bf.expand(islands, pad=2000), islands))
print(len(islands), "islands ->", len(shores), "shores")
shores.head()

## Both on the genome

One `add_features` per frame. Islands are colored by GC% — a column that rides along and shows in each feature's details; any column does. This lands on *TP53*.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, make_assembly

hg38 = make_assembly(
    "hg38",
    "https://jbrowse.org/genomes/GRCh38/fasta/hg38.prefix.fa.gz",
    aliases=["GRCh38"],
)
view = LinearGenomeView(assembly=hg38, location="17:7,660,000..7,700,000")
view.add_features(
    islands, name="CpG islands (by GC%)",
    color="jexl:get(feature,'perGc') > 65 ? '#00695c' : '#4db6ac'",
)
view.add_features(shores, name="CpG shores", color="#f9a825")
view